In [1]:
import asyncio
import selectors
import sys
from pathlib import Path
from threading import Thread

from dotenv import load_dotenv
from IPython import get_ipython

# 从 Notebook 所在目录定位项目，并在导入配置前加载环境变量。
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "app" / "core" / "config.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("未找到包含 app/core/config.py 的项目目录")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
load_dotenv(project_root / ".env")

# 重复运行时先关闭旧连接池。
_previous_pool = globals().get("checkpoint_pool")
if _previous_pool is not None and not _previous_pool.closed:
    await _previous_pool.close()

# Windows Notebook 的异步单元格共用兼容 Psycopg 的 Selector 循环。
if sys.platform == "win32":
    _postgres_loop = globals().get("_postgres_loop")
    if _postgres_loop is None or _postgres_loop.is_closed():
        _postgres_loop = asyncio.SelectorEventLoop(selectors.SelectSelector())
        _postgres_thread = Thread(
            target=_postgres_loop.run_forever,
            name="postgres-notebook",
            daemon=True,
        )
        _postgres_thread.start()

    def _run_postgres(coro):
        return asyncio.run_coroutine_threadsafe(coro, _postgres_loop).result()

    _shell = get_ipython()
    if _shell is None:
        raise RuntimeError("请在 Jupyter Notebook 中运行此单元格")
    _shell.loop_runner = _run_postgres
    _shell.autoawait = True


In [3]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

from psycopg.rows import dict_row
from psycopg_pool import AsyncConnectionPool
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from app.core.config import settings
from dotenv import load_dotenv

load_dotenv()

# 创建连接池
checkpoint_pool = AsyncConnectionPool(
    conninfo=settings.database.checkpoint_url,
    min_size=1,  # 池最小连接数
    max_size=5,  # 池最大连接数
    kwargs={
        "autocommit": True,  # 自动提交事务
        "prepare_threshold": 0,  # 不做预准备sql
        "row_factory": dict_row,  # 查询结果以dict返回
    },
    open=False,  # 不自动创建连接
)
# 初始化连接
await checkpoint_pool.open()
# 等待连接池就绪
await checkpoint_pool.wait()

# 初始化checkpointer
checkpointer = AsyncPostgresSaver(checkpoint_pool)
# 自动建表（与checkpointer存储有关的表）
await checkpointer.setup()


# 初始化模型
model = init_chat_model(
    "deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}},
)
# 创建Agent
agent = create_agent(
    model=model,
    tools=[],
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "day06-checkpointer-test"}}
# 第一次，告知自己的信息
result = await agent.ainvoke(
    {"messages": [HumanMessage(content="我叫小明，今年30岁。")]},
    config=config,
)
# 第二次，询问信息
result = await agent.ainvoke(
    {"messages": [HumanMessage(content="我今年多大？")]},
    config=config,
)
print(result["messages"][-1].content)


config = {"configurable": {"thread_id": "52367f2f-1e96-494c-9112-83bd1817fb9f"}}

snapshot = await agent.aget_state(config)

for message in snapshot.values.get("messages", []):
    print(f"{message.type}: {message.content}")

你今年 **30岁**。😊

这是你告诉我的信息：“我叫小明，今年30岁。”
human: 在的吗
ai: 您好，我在的。我是您的智能保险顾问，很高兴为您服务。请问有什么可以帮您？
human: 我是你爸爸，记住没有
ai: 您好，我理解您可能在开玩笑，不过我这边是保险咨询顾问，主要负责解答保险相关问题。如果您有保险方面的疑问，欢迎随时告诉我，我会认真帮您分析。
human: 我刚刚说了什么
ai: 您刚才说：“我是你爸爸，记住没有”。

如果您是随口一说，没关系；我这边主要还是帮您解答保险相关问题。您有想咨询的保险内容吗？
